In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import requests
import os
from pprint import pprint as display
 # Import userdata



headers = {
    'Content-Type': 'application/json',
    'X-Goog-Api-Key': os.getenv("GOOGLE_MAPS"), # Use userdata.get
    'X-Goog-FieldMask': 'places.id,places.displayName,places.formattedAddress,places.priceLevel'

}

req = requests.post(
    "https://places.googleapis.com/v1/places:searchText",
    json={"textQuery" : "Spicy Vegetarian Food in Sydney, Australia"},
    headers=headers
)

display(req.json())

{'places': [{'displayName': {'languageCode': 'en', 'text': 'Peace Harmony'},
             'formattedAddress': '29 King St, Sydney NSW 2000, Australia',
             'id': 'ChIJs5ydyTiuEmsR0fRSlU0C7k0',
             'priceLevel': 'PRICE_LEVEL_INEXPENSIVE'},
            {'displayName': {'languageCode': 'en',
                             'text': 'Tian Ci Vegan (Takeaway only)'},
             'formattedAddress': '101 Cleveland St, Darlington NSW 2008, '
                                 'Australia',
             'id': 'ChIJUwJPSHOxEmsRlxW9EbU2cpg',
             'priceLevel': 'PRICE_LEVEL_INEXPENSIVE'},
            {'displayName': {'languageCode': 'en', 'text': 'The Spice Room'},
             'formattedAddress': 'The Quay Building, 2 Phillip St, Sydney NSW '
                                 '2000, Australia',
             'id': 'ChIJE18nTWiuEmsRUz-zCmu8p8U',
             'priceLevel': 'PRICE_LEVEL_MODERATE'},
            {'displayName': {'languageCode': 'en',
                             'te

In [ ]:
headers = {
    'Content-Type': 'application/json',
    'X-Goog-Api-Key': 'nope',
    'X-Goog-FieldMask':"id,displayName,formattedAddress,internationalPhoneNumber,websiteUri"}
req=requests.get("https://places.googleapis.com/v1/places/ChIJMWdcBCeuEmsRpr9CDsS4-nc",headers=headers)
req.json()

{'id': 'ChIJMWdcBCeuEmsRpr9CDsS4-nc',
 'internationalPhoneNumber': '+61 2 9281 0822',
 'formattedAddress': 'Kensington St, Chippendale NSW 2008, Australia',
 'websiteUri': 'http://spicealley.com.au/',
 'displayName': {'text': 'Spice Alley', 'languageCode': 'en'}}

In [2]:
!pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.6 MB/s eta 0:00:00


In [6]:
# lead_enricher.py
import os, time, csv, re, json, tldextract, html
import requests
from urllib.parse import urlencode, urljoin
from urllib import robotparser
from bs4 import BeautifulSoup

GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_MAPS") or "YOUR_GOOGLE_PLACES_KEY"
BING_API_KEY = os.getenv("BING_API_KEY")  # optional, for LinkedIn fallback

UA = os.getenv("USER_AGENT") or "LeadFinderBot/1.0 (+contact@example.com)"
REQ_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT") or "12")

EMAIL_RE = re.compile(r"""
    (?<![\w.+-])                # left boundary
    [A-Z0-9._%+\-]+
    \s*(?:\[at\]|\(at\)|@|\s+at\s+)\s*
    [A-Z0-9.\-]+
    \s*(?:\[dot\]|\(dot\)|\.|\s+dot\s+)\s*
    [A-Z]{2,20}
""", re.I | re.X)

CLEANERS = [
    (re.compile(r"\s*\(at\)\s*|\s*\[at\]\s*|\s+at\s+", re.I), "@"),
    (re.compile(r"\s*\(dot\)\s*|\s*\[dot\]\s*|\s+dot\s+", re.I), "."),
    (re.compile(r"\s+"), ""),
]

FREE_DOMAINS = {
    "gmail.com","yahoo.com","outlook.com","hotmail.com","aol.com","icloud.com",
    "proton.me","protonmail.com","yandex.com","zoho.com"
}

def google_places_search_text(text_query, field_mask):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
        "X-Goog-FieldMask": field_mask,
    }
    resp = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        json={"textQuery": text_query},
        headers=headers, timeout=REQ_TIMEOUT)
    resp.raise_for_status()
    return resp.json().get("places", [])

def google_place_details(place_resource_name, field_mask):
    # place_resource_name is like "places/ChIJxxxxx..."
    headers = {
        "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
        "X-Goog-FieldMask": field_mask,
    }
    url = f"https://places.googleapis.com/v1/{place_resource_name}"
    r = requests.get(url, headers=headers, timeout=REQ_TIMEOUT)
    r.raise_for_status()
    return r.json()

def can_fetch(robots_url, url, ua=UA):
    try:
        rp = robotparser.RobotFileParser()
        rp.set_url(robots_url); rp.read()
        return rp.can_fetch(ua, url)
    except Exception:
        return True  # be permissive if robots unreachable

def fetch(url):
    r = requests.get(url, headers={"User-Agent": UA}, timeout=REQ_TIMEOUT)
    r.raise_for_status()
    return r.text

def candidate_paths(base):
    base = base.rstrip("/")
    return [base, base+"/contact", base+"/contact-us", base+"/about", base+"/team", base+"/support"]

def normalize_email(raw):
    s = html.unescape(raw)
    for pat, repl in CLEANERS:
        s = pat.sub(repl, s)
    s = s.strip().strip(".,;:()[]{}<>")
    # final strict email
    m = re.search(r"[A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,}", s, re.I)
    return m.group(0).lower() if m else None

def extract_emails_from_html(html_text):
    found = set()
    """for m in EMAIL_RE.findall(html_text):
        e = normalize_email(m)
        if e:
            found.add(e)"""
    # also catch mailto links
    soup = BeautifulSoup(html_text, "html.parser")
    for a in soup.select('a[href^="mailto:"]'):
        raw = a.get("href","")[7:]
        e = normalize_email(raw)
        if e:
            found.add(e)
    return sorted(found)

def likely_company_emails(emails, site_domain):
    out = []
    for e in emails:
        dom = e.split("@")[-1]
        if dom in FREE_DOMAINS:
            continue
        if site_domain and site_domain not in dom:
            # keep but deprioritize; you can choose to filter out
            pass
        out.append(e)
    return sorted(set(out))

def find_linkedin_on_site(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "linkedin.com/company/" in href or "linkedin.com/in/" in href:
            return href.split("?")[0]
    return None

def bing_find_linkedin(domain_or_name):
    if not BING_API_KEY:
        return None
    q = f'site:linkedin.com/company "{domain_or_name}"'
    url = f"https://api.bing.microsoft.com/v7.0/search?{urlencode({'q': q, 'count': 5})}"
    r = requests.get(url, headers={"Ocp-Apim-Subscription-Key": BING_API_KEY}, timeout=REQ_TIMEOUT)
    if r.status_code != 200:
        return None
    data = r.json()
    for item in (data.get("webPages") or {}).get("value", []):
        u = item.get("url","")
        if "linkedin.com/company/" in u:
            return u.split("?")[0]
    return None

def enrich_site(website):
    if not website:
        return [], None
    extracted = tldextract.extract(website)
    reg_domain = f"{extracted.domain}.{extracted.suffix}" if extracted.suffix else extracted.domain
    base = website if website.startswith("http") else "http://" + website
    robots_url = urljoin(base, "/robots.txt")
    emails, linkedin = set(), None
    for url in candidate_paths(base):
        try:
            if not can_fetch(robots_url, url):
                continue
            html_text = fetch(url)
            emails |= set(extract_emails_from_html(html_text))
            if not linkedin:
                linkedin = find_linkedin_on_site(html_text)
            time.sleep(0.5)
        except Exception:
            continue
    filtered_emails = likely_company_emails(sorted(emails), reg_domain)
    if not linkedin:
        # try Bing fallback using domain
        linkedin = bing_find_linkedin(reg_domain)
    return filtered_emails, linkedin

def run_pipeline(keyword, location, out_csv="leads.csv", max_places_pages=1):
    # 1) search text -> get basic list with resource names
    field_mask_search = "places.name,places.id,places.displayName,places.formattedAddress"
    places = google_places_search_text(f"{keyword} in {location}", field_mask_search)

    # 2) fetch details to get website + phone
    field_mask_details = "id,displayName,formattedAddress,internationalPhoneNumber,websiteUri"
    rows = []
    for i, p in enumerate(places, 1):
        name = (p.get("displayName") or {}).get("text")
        addr = p.get("formattedAddress")
        resource_name = p.get("name") or p.get("id")
        if not resource_name:
            continue
        try:
            det = google_place_details(resource_name, field_mask_details)
        except Exception:
            continue
        website = det.get("websiteUri")
        phone = det.get("internationalPhoneNumber")
        emails, linkedin = enrich_site(website)
        rows.append({
            "name": name,
            "address": addr,
            "phone": phone,
            "website": website,
            "emails": ";".join(emails),
            "linkedin": linkedin
        })
        print(f"[{i}/{len(places)}] {name} -> emails:{len(emails)} linkedin:{bool(linkedin)}")
        time.sleep(0.2)

    # 3) write CSV
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["name","address","phone","website","emails","linkedin"])
        w.writeheader()
        for r in rows:
            w.writerow(r)
    print(f"Wrote {len(rows)} rows -> {out_csv}")

if __name__ == "__main__":
    # Example:
    #   setenv GOOGLE_PLACES_API_KEY=...
    #   optional: setenv BING_API_KEY=...
    #   python lead_enricher.py
    run_pipeline(keyword="Cricket Goods Store", location="Sydney, Australia", out_csv="leads.csv")


[1/20] GA Sports -> emails:0 linkedin:False


KeyboardInterrupt: 

In [5]:
import pandas as pd

df=pd.read_csv("leads.csv")
df['emails'][1]

nan

In [13]:
import os
import re
import time
import html
import tldextract
import requests
from urllib.parse import urlencode, urlparse
#from google.colab import userdata # Import userdata
import math # Import math to check for NaN

# ---------- config ----------
# Use userdata.get for secure storage of API keys in Colab
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY") or os.getenv("TAVILY_API_KEY") or "YOUR_TAVILY_KEY"
UA = os.getenv("USER_AGENT") or "LeadFinderBot/1.0 (+contact@example.com)"
REQ_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT") or "12")
BACKOFF = 0.5  # polite delay between fetches

FREE_DOMAINS = {
    "gmail.com","yahoo.com","outlook.com","hotmail.com","aol.com","icloud.com",
    "proton.me","protonmail.com","yandex.com","zoho.com"
}

EMAIL_FUZZY = re.compile(r"""
    (?<![\w.+-])
    [A-Z0-9._%+\-]+
    \s*(?:\[at\]|\(at\)|@|\s+at\s+)\s*
    [A-Z0-9.\-]+
    \s*(?:\[dot\]|\(dot\)|.|\s+dot\s+)\s*
    [A-Z]{2,24}
""", re.I | re.X)

# ---------- helpers ----------
def _normalize_email(raw: str) -> str | None:
    s = html.unescape(raw)
    s = re.sub(r"\s*\(at\)\s*|\s*\[at\]\s*|\s+at\s+", "@", s, flags=re.I)
    s = re.sub(r"\s*\(dot\)\s*|\s*\[dot\)\s*|\s+dot\s+", ".", s, flags=re.I)
    s = s.strip().strip(".,;:()[]{}<>")
    m = re.search(r"[A-Z0-9._%+\-]+@[A-Z0-9.\-]+\.[A-Z]{2,}", s, re.I)
    return m.group(0).lower() if m else None

def _extract_linkedins(text: str) -> set[str]:
    found = set()
    for m in re.findall(r"https?://(www\.)?linkedin\.com/(in|company)/[A-Za-z0-9\-_]+", text or "", flags=re.I):
        url = "https://linkedin.com/" + "/".join(m[1:])
        found.add(url.split("?")[0])  # strip tracking
    return found

def _extract_emails(text: str) -> set[str]:
    found = set()
    for m in EMAIL_FUZZY.findall(text or ""):
        e = _normalize_email(m)
        if e:
            found.add(e)
    # Also check mailto: links if HTML provided
    for mailto in re.findall(r'href=["\']mailto:([^"\']+)["\']', text or "", flags=re.I):
        e = _normalize_email(mailto)
        if e:
            found.add(e)
    return found

def _is_company_email(email: str, company_domain: str | None) -> bool:
    dom = email.split("@")[-1].lower()
    if dom in FREE_DOMAINS:
        return False
    return (company_domain is None) or (company_domain in dom)

def _rank_email(email: str) -> int:
    local = email.split("@")[0].lower()
    # preference for non-generic mailboxes
    if local in {"info","hello","contact","support","admin","sales"}:
        return 10
    if re.fullmatch(r"[a-z]+\.[a-z]+", local) or re.fullmatch(r"[a-z]+", local):
        return 50  # looks like a person
    return 30

def _company_domain_from_website(website: str | None) -> str | None:
    if not website:
        return None
    ext = tldextract.extract(website)
    if not ext.suffix:
        return None
    return f"{ext.domain}.{ext.suffix}".lower()

def _fetch(url: str) -> str:
    r = requests.get(url, headers={"User-Agent": UA}, timeout=REQ_TIMEOUT)
    r.raise_for_status()
    return r.text

def _tavily_search(query: str, max_results: int = 5) -> list[dict]:
    # REST call (no extra SDK dependency)
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "search_depth": "advanced",
        "max_results": max_results,
        "include_answer": False,
        "include_images": False,
        "include_domains": None,   # you can set to [domain] to restrict
    }
    r = requests.post(url, json=payload, timeout=REQ_TIMEOUT, headers={"User-Agent": UA})
    r.raise_for_status()
    data = r.json()
    return data.get("results", []) or []

# ---------- main enrichment ----------
def tavily_find_emails(name: str,
                       website: str | None = None,
                       location: str | None = None,
                       max_results_per_query: int = 5,
                       fetch_pages: bool = True) -> dict:
    """
    Use Tavily web search to discover company emails for a place where initial crawl failed.
    Returns dict: {"emails": [..sorted..], "sources": {email: [urls...]}, "queries": [..]}
    """
    company_domain = _company_domain_from_website(website)
    q_base = name.strip()
    if location:
        q_base += f" {location.strip()}"

    queries = []
    # High-signal queries
    if company_domain:
        queries += [
            f'site:{company_domain} (email OR contact OR support OR sales)',
            f'"@{company_domain}" email',
            f'"@{company_domain}" LinkedIn',
        ]
    # Brand/name oriented
    queries += [
        f'{name} "contact us" email',
        f'{name} support email',
        f'{name} Staff LinkedIn Profile',

    ]
    if location:
        queries += [
            f'{name} email {location}',
            f'{name} "contact" {location}',
        ]

    found_emails: set[str] = set()
    found_linkedins: set[str] = set()
    sources: dict[str, set[str]] = {}

    for q in queries:
        try:
            results = _tavily_search(q, max_results=max_results_per_query)
        except Exception:
            continue

        # Extract from Tavily snippets first
        for res in results:
            content = f"{res.get('title','')}\n{res.get('content','')}\n{res.get('url','')}"
            for e in _extract_emails(content):
                found_emails.add(e)
                sources.setdefault(e, set()).add(res.get("url",""))
            for l in _extract_linkedins(content):
                found_linkedins.add(l)
                sources.setdefault(l, set()).add(res.get("url",""))

        # Optionally fetch top result pages for deeper extraction
        if fetch_pages:
            for res in results[:max(2, max_results_per_query // 2)]:  # keep it light
                url = res.get("url")
                if not url:
                    continue
                # Skip obvious social/aggregators
                netloc = urlparse(url).netloc.lower()
                if any(s in netloc for s in ("facebook.com","instagram.com","x.com","twitter.com","linkedin.com")):
                    continue
                try:
                    html_text = _fetch(url)
                    emails_in_page = _extract_emails(html_text)
                    for e in emails_in_page:
                        found_emails.add(e)
                        sources.setdefault(e, set()).add(url)
                    time.sleep(BACKOFF)
                except Exception:
                    continue

        # Early exit if we’ve already got good company-domain emails
        if any(_is_company_email(e, company_domain) for e in found_emails):
            break

    # Filter and rank
    filtered = [e for e in found_emails if _is_company_email(e, company_domain)]
    # If none on company domain, fall back to any non-free emails
    if not filtered:
        filtered = [e for e in found_emails if e.split("@")[-1].lower() not in FREE_DOMAINS]

    ranked = sorted(filtered, key=_rank_email, reverse=True)
    return {
        "emails": ranked,
        "linkedins": sorted(found_linkedins),
        "sources": {k: sorted(v) for k, v in sources.items() if k in ranked},
        "queries": queries,
        "company_domain": company_domain
    }

# ---------- example ----------


In [14]:
import pandas as pd
df=pd.read_csv("leads_from_api.csv")
website_param = df['website'][1] if isinstance(df['website'][1], str) and not (isinstance(df['website'][1], float) and math.isnan(df['website'][1])) else None
result = tavily_find_emails(
                    name=df["name"][1],
                    website=website_param,
                    location=df['address'][1],
                )
print(result)

{'emails': ['customerservice@amgosports.com.au'], 'linkedins': [], 'sources': {'customerservice@amgosports.com.au': ['https://amgosports.com.au/pages/contact?srsltid=AfmBOooF67-GXmkF1LA-fq9EbUfMGKPx4PIhclxw3s1RFxQqZJBpwkcG', 'https://amgosports.com.au/pages/contact?srsltid=AfmBOopqv7gIaqh3KRXaG-VuBe8G3qkMDKXCcFDi9DCnWno6ZvfUnZKp']}, 'queries': ['site:amgosports.com.au (email OR contact OR support OR sales)', '"@amgosports.com.au" email', '"@amgosports.com.au" LinkedIn', 'AMGO SPORTS Cricket Store "contact us" email', 'AMGO SPORTS Cricket Store support email', 'AMGO SPORTS Cricket Store Staff LinkedIn Profile', 'AMGO SPORTS Cricket Store email 30 Tuckwell Rd, Castle Hill NSW 2154, Australia', 'AMGO SPORTS Cricket Store "contact" 30 Tuckwell Rd, Castle Hill NSW 2154, Australia'], 'company_domain': 'amgosports.com.au'}


Trying to set up Email Sending

In [ ]:
import smtplib, ssl
from email.message import EmailMessage

def send_email(
    smtp_host: str,
    smtp_port: int,
    smtp_user: str,
    smtp_password: str,
    from_addr: str,
    to_addr: str,
    subject: str,
    body: str,
    is_html: bool = False
):
    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"] = from_addr
    msg["To"] = to_addr

    if is_html:
        msg.set_content("This is an HTML email. Your client does not support HTML.")
        msg.add_alternative(body, subtype="html")
    else:
        msg.set_content(body)

    context = ssl.create_default_context()
    # If using SSL (port 465)
    with smtplib.SMTP_SSL(smtp_host, smtp_port, context=context) as server:
        server.login(smtp_user, smtp_password)
        server.send_message(msg)

# Usage example
if __name__ == "__main__":
    send_email(
        smtp_host="smtp.gmail.com",
        smtp_port=465,
        smtp_user="souravsahil100@gmail.com",
        smtp_password="pfed vgyb hile qnlo",
        subject="Test from Python",
        body="bals testing 123"
    )


SMTPAuthenticationError: (535, b'5.7.8 Username and Password not accepted. For more information, go to\n5.7.8  https://support.google.com/mail/?p=BadCredentials 98e67ed59e1d1-33bb6626309sm3817420a91.7 - gsmtp')

In [3]:
from tavily_enricher import tavily_enrich_lead



# Test Tavily Enrichment

Test the `tavily_enrich_lead` function with a sample company to find emails and LinkedIn profiles.

In [5]:
# Test Case: R.R. Interior from leads_from_api.csv
# This is a real company with NO website, NO emails, NO LinkedIn
print("=" * 80)
print("TEST CASE: R.R. Interior (Real lead from CSV - no existing data)")
print("=" * 80)

result = tavily_enrich_lead(
    name="R.R. Interior",
    website=None,  # No website available
    location="Police chowki, DLF Gardencity Enclave Rd, opposite to Hayatpur, Hayatpur, Sector 94, Gurugram, Haryana 122505, India",
    existing_emails=[],  # No existing emails
    existing_linkedin=None,  # No existing LinkedIn
    max_queries=3,  # Allow up to 3 queries
    fetch_pages=False  # Don't fetch pages for faster test
)

print(f"\n📋 Company Details:")
print(f"  Name: R.R. Interior")
print(f"  Location: Gurugram, Haryana, India")
print(f"  Phone: +91 80000 08017")
print(f"  Website: None")
print(f"  Existing emails: None")
print(f"  Existing LinkedIn: None")

print(f"\n🔍 Tavily Enrichment Results:")
print(f"  ✉️  Emails found: {len(result['emails'])}")
if result['emails']:
    for i, email in enumerate(result['emails'][:3], 1):  # Show top 3
        print(f"      {i}. {email}")
        if email in result.get('sources', {}):
            sources = result['sources'][email]
            print(f"         Sources: {', '.join(sources[:2])}")  # Show first 2 sources
else:
    print(f"      ❌ No emails found")

print(f"\n  🔗 LinkedIn: {result['linkedin'] if result['linkedin'] else '❌ Not found'}")

print(f"\n💰 Cost Information:")
print(f"  Queries executed: {result['queries_run']}")
print(f"  API calls made: {result['api_calls_made']}")
print(f"  Estimated cost: ${result['cost_estimate_usd']}")

print(f"\n🎯 Company Domain: {result['company_domain'] if result['company_domain'] else 'N/A (no website)'}")

if 'skipped' in result:
    print(f"\n⚠️  Status: {result['skipped']}")

print("\n" + "=" * 80)
print("✅ Test Complete!")
print("=" * 80)

TEST CASE: R.R. Interior (Real lead from CSV - no existing data)
[DEBUG] tavily_enrich_lead called with: {'name': 'R.R. Interior', 'website': None, 'location': 'Police chowki, DLF Gardencity Enclave Rd, opposite to Hayatpur, Hayatpur, Sector 94, Gurugram, Haryana 122505, India', 'existing_emails': [], 'existing_linkedin': None, 'max_queries': 3, 'fetch_pages': False}

📋 Company Details:
  Name: R.R. Interior
  Location: Gurugram, Haryana, India
  Phone: +91 80000 08017
  Website: None
  Existing emails: None
  Existing LinkedIn: None

🔍 Tavily Enrichment Results:
  ✉️  Emails found: 2
      1. chaitali.sarang@innovarip.com
         Sources: https://ipindia.gov.in/writereaddata/Portal/Images/pdf/Trade_Marks_Agents_List__31-12-2024.pdf
      2. accounts@ksrandco.in
         Sources: https://ipindia.gov.in/writereaddata/Portal/Images/pdf/Trade_Marks_Agents_List__31-12-2024.pdf

  🔗 LinkedIn: https://linkedin.com/company

💰 Cost Information:
  Queries executed: ['R.R. Interior contact emai